# Progressive Expansion Pipeline

**What this does:** Starts from a single medical term and iteratively expands to find all related SNOMED CT concepts through multiple search strategies.

## Pipeline Stages

| Stage | Method | Description |
|-------|--------|-------------|
| **1** | Term Matching | Finds exact and prefix matches in SNOMED descriptions |
| **2** | Hierarchy Expansion | Traverses parent-child relationships (breadth-first) |
| **3** | Embedding Similarity | FAISS semantic search using LLM embeddings |
| **4** | MedCAT Co-occurrence | Finds concepts appearing together in clinical text |
| **5** | LLM Generation | Uses Qwen2.5-Coder to suggest novel related terms |

## Results show:
- CUI (unique concept identifier)
- Preferred name (human-readable term)
- Which stages found each concept


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../src"))
from snomed_methods import ProgressiveExpansionPipeline

print("OK")

In [ ]:
# Initialize pipeline with data sources

pipeline = ProgressiveExpansionPipeline(
    uk_path="../uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z",
    medcat_path="../model_packs/medcat_model_pack_422d1d38fc58f158.zip",
    model_path="../embedding_models/SapBERT-from-PubMedBERT-fulltext",
    backend="transformers",
)

print("Pipeline initialized with UK Clinical RF2, MedCAT, and SapBERT")

In [ ]:
# =====================================================
# EXAMPLE 1: Meningioma
# =====================================================

result = pipeline.expand(
    query="meningioma",
    stages=["term", "hierarchy", "embedding"],
    max_concepts=50,
)

concepts = result.get_concepts_with_names()

print("Query: 'meningioma'")
print(f"Found {len(concepts)} total concepts")
print("\nTop concepts (CUI | Preferred Name):")
for i, (cui, name) in enumerate(concepts[:15], 1):
    print(f"{i:2}. [{cui}] {name}")

In [ ]:
# Show evidence breakdown

print("=" * 60)
print("EVIDENCE BREAKDOWN")
print("=" * 60)

for stage in result.stages_executed:
    cuis = result.sources[stage]
    score = result.scores.get(stage, 0)
    print(f"{stage:15} : Added {len(cuis):3} concepts (score={score:.3f})")

print("\nTotal unique concepts:", len(result.all_cuis))

In [ ]:
# =====================================================
# EXAMPLE 2: HAEMOCHROMATOSIS - Method Effectiveness
# =====================================================

# Hand-picked real haemochromatosis concepts from MedCAT CDB for validation
# These are the concepts that should be found by our pipeline
known_hemo_concepts = [
    "Hereditary haemochromatosis",
    "Haemochromatosis gene screening test",
    "Carrier of haemochromatosis HFE gene mutation (finding)",
    "Primary haemochromatosis",
    "Secondary haemochromatosis",
    "Hypoparathyroidism due to haemochromatosis",
]

# Expand from 'haemochromatosis' query
result_hemo = pipeline.expand(
    query="haemochromatosis",
    stages=["term", "hierarchy", "medcat"],
    max_concepts=100,
)

# Get concepts with names
found_concepts = result_hemo.get_concepts_with_names()

print("Query: 'haemochromatosis'")
print(f"Found {len(found_concepts)} SNOMED concepts via expansion\n")

# Check which known concepts were found
found_terms = []
not_found_terms = []

for term in known_hemo_concepts:
    matched = False
    for cui, name in found_concepts:
        if term.lower() in name.lower():
            found_terms.append((term, cui, name))
            matched = True
            break
    if not matched:
        not_found_terms.append(term)

print("=" * 60)
print(
    f"Known haemochromatosis concepts found: {len(found_terms)}/{len(known_hemo_concepts)}",
)
print("=" * 60)
for term, cui, name in found_terms:
    print(f"✓ '{term}' -> [{cui}] {name}")

if not_found_terms:
    print(f"\nNot found ({len(not_found_terms)}):")
    for term in not_found_terms:
        print(f"  - '{term}' (may require different SNOMED edition)")